In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, ConfusionMatrixDisplay, f1_score, accuracy_score
)
from sklearn.utils import resample

# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────

In [2]:
print("=" * 70)
print("STEP 1: LOADING DATA")
print("=" * 70)

df = pd.read_csv('student_data.csv', sep=';')

# Clean column names
df.columns = df.columns.str.strip().str.replace('\t', '', regex=False)
df.rename(columns={df.columns[0]: df.columns[0].lstrip('\ufeff')}, inplace=True)

print(f"Dataset shape: {df.shape}")
print(f"\nTarget distribution:")
print(df['Target'].value_counts())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"\nFeature types:\n{df.dtypes.value_counts()}")


STEP 1: LOADING DATA
Dataset shape: (4424, 37)

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Missing values: 0

Feature types:
int64      29
float64     7
object      1
Name: count, dtype: int64


# ─────────────────────────────────────────────────────────────────────────────
# 2. EXPLORATORY DATA ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
print("\n" + "=" * 70)
print("STEP 2: EXPLORATORY DATA ANALYSIS")
print("=" * 70)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Exploratory Data Analysis', fontsize=16, fontweight='bold', y=1.02)

# 2a. Class distribution
colors = ['#e74c3c', '#2ecc71', '#3498db']
counts = df['Target'].value_counts()
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Class Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', fontsize=10)
axes[0].set_ylim(0, max(counts.values) * 1.2)

# 2b. Age at enrollment by target
dropout_ages = df[df['Target'] == 'Dropout']['Age at enrollment']
grad_ages    = df[df['Target'] == 'Graduate']['Age at enrollment']
enr_ages     = df[df['Target'] == 'Enrolled']['Age at enrollment']
axes[1].hist([dropout_ages, grad_ages, enr_ages], bins=25, color=colors,
             label=['Dropout', 'Graduate', 'Enrolled'], alpha=0.75, edgecolor='white')
axes[1].set_title('Age at Enrollment by Outcome', fontweight='bold')
axes[1].set_xlabel('Age')
axes[1].set_ylabel('Count')
axes[1].legend()

# 2c. 2nd sem grade by target
for i, (label, color) in enumerate(zip(['Dropout', 'Graduate', 'Enrolled'], colors)):
    subset = df[df['Target'] == label]['Curricular units 2nd sem (grade)']
    axes[2].hist(subset, bins=20, alpha=0.65, color=color, label=label, edgecolor='white')
axes[2].set_title('2nd Semester Grade by Outcome', fontweight='bold')
axes[2].set_xlabel('Grade')
axes[2].set_ylabel('Count')
axes[2].legend()

plt.tight_layout()
plt.savefig('plot_eda.png', dpi=150, bbox_inches='tight')
plt.close()
print("EDA plots saved.")



STEP 2: EXPLORATORY DATA ANALYSIS
EDA plots saved.


# ─────────────────────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────────────────────

In [4]:
print("\n" + "=" * 70)
print("STEP 3: FEATURE ENGINEERING")
print("=" * 70)

df_fe = df.copy()

# Academic efficiency: ratio of approved to enrolled units (each semester)
df_fe['approval_rate_sem1'] = np.where(
    df_fe['Curricular units 1st sem (enrolled)'] > 0,
    df_fe['Curricular units 1st sem (approved)'] / df_fe['Curricular units 1st sem (enrolled)'],
    0
)
df_fe['approval_rate_sem2'] = np.where(
    df_fe['Curricular units 2nd sem (enrolled)'] > 0,
    df_fe['Curricular units 2nd sem (approved)'] / df_fe['Curricular units 2nd sem (enrolled)'],
    0
)

# Grade trajectory: improvement/decline between semesters
df_fe['grade_delta'] = (
    df_fe['Curricular units 2nd sem (grade)'] -
    df_fe['Curricular units 1st sem (grade)']
)

# Financial stress flag
df_fe['financial_stress'] = (
    (df_fe['Debtor'] == 1) | (df_fe['Tuition fees up to date'] == 0)
).astype(int)

# Total approved units across both semesters
df_fe['total_approved'] = (
    df_fe['Curricular units 1st sem (approved)'] +
    df_fe['Curricular units 2nd sem (approved)']
)

# Average grade across both semesters
df_fe['avg_grade'] = (
    df_fe['Curricular units 1st sem (grade)'] +
    df_fe['Curricular units 2nd sem (grade)']
) / 2

print("New features added:")
new_feats = ['approval_rate_sem1', 'approval_rate_sem2', 'grade_delta',
             'financial_stress', 'total_approved', 'avg_grade']
for f in new_feats:
    print(f"  + {f}")



STEP 3: FEATURE ENGINEERING
New features added:
  + approval_rate_sem1
  + approval_rate_sem2
  + grade_delta
  + financial_stress
  + total_approved
  + avg_grade


# ─────────────────────────────────────────────────────────────────────────────
# 4. PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

In [5]:
print("\n" + "=" * 70)
print("STEP 4: PREPROCESSING")
print("=" * 70)

# Encode target
le = LabelEncoder()
df_fe['target_encoded'] = le.fit_transform(df_fe['Target'])
print(f"Class encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Features and target
X = df_fe.drop(columns=['Target', 'target_encoded'])
y = df_fe['target_encoded']

# Scale features
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Train/test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Train class distribution: {pd.Series(y_train).value_counts().to_dict()}")
print(f"Test class distribution:  {pd.Series(y_test).value_counts().to_dict()}")

# Handle class imbalance via oversampling (resampling minority classes to match majority)
y_train_reset = y_train.reset_index(drop=True)
train_df = X_train.reset_index(drop=True).copy()
train_df['__label__'] = y_train_reset

majority_class = train_df['__label__'].value_counts().idxmax()
majority_count = train_df['__label__'].value_counts().max()

balanced_parts = []
for cls in train_df['__label__'].unique():
    subset = train_df[train_df['__label__'] == cls]
    if cls != majority_class:
        subset = resample(subset, replace=True, n_samples=majority_count, random_state=42)
    balanced_parts.append(subset)

train_balanced = pd.concat(balanced_parts).sample(frac=1, random_state=42)
X_train_bal = train_balanced.drop(columns=['__label__'])
y_train_bal  = train_balanced['__label__']

print(f"\nAfter balancing - Train class distribution: {pd.Series(y_train_bal).value_counts().to_dict()}")



STEP 4: PREPROCESSING
Class encoding: {'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}

Train size: 3539 | Test size: 885
Train class distribution: {2: 1767, 0: 1137, 1: 635}
Test class distribution:  {2: 442, 0: 284, 1: 159}

After balancing - Train class distribution: {1: 1767, 0: 1767, 2: 1767}


# ─────────────────────────────────────────────────────────────────────────────
# 5. MODEL TRAINING
# ─────────────────────────────────────────────────────────────────────────────

In [6]:
print("\n" + "=" * 70)
print("STEP 5: MODEL TRAINING")
print("=" * 70)

models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, C=1.0,
        class_weight='balanced', random_state=42
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=20,
        class_weight='balanced', random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_leaf=10,
        class_weight='balanced', n_jobs=-1, random_state=42
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, random_state=42
    )
}

trained_models = {}
cv_scores = {}

for name, model in models.items():
    print(f"\n  Training {name}...", end=' ')
    model.fit(X_train_bal, y_train_bal)
    trained_models[name] = model

    # 5-fold cross-validation on balanced training data
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_f1 = cross_val_score(model, X_train_bal, y_train_bal,
                             cv=cv, scoring='f1_weighted', n_jobs=-1)
    cv_scores[name] = cv_f1
    print(f"CV F1 (weighted): {cv_f1.mean():.4f} ± {cv_f1.std():.4f}")



STEP 5: MODEL TRAINING

  Training Logistic Regression... CV F1 (weighted): 0.7240 ± 0.0112

  Training Decision Tree... CV F1 (weighted): 0.7292 ± 0.0152

  Training Random Forest... CV F1 (weighted): 0.8108 ± 0.0168

  Training Gradient Boosting... CV F1 (weighted): 0.8999 ± 0.0110


# ─────────────────────────────────────────────────────────────────────────────
# 6. EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

In [7]:
print("\n" + "=" * 70)
print("STEP 6: EVALUATION ON TEST SET")
print("=" * 70)

results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)

    acc  = accuracy_score(y_test, y_pred)
    f1_w = f1_score(y_test, y_pred, average='weighted')
    f1_m = f1_score(y_test, y_pred, average='macro')
    # ROC-AUC (OvR, macro)
    roc  = roc_auc_score(y_test, y_prob, multi_class='ovr', average='macro')

    # Per-class F1 for dropout (class index from LabelEncoder)
    dropout_idx = list(le.classes_).index('Dropout')
    f1_dropout = f1_score(y_test, y_pred, average=None)[dropout_idx]

    results[name] = {
        'Accuracy': acc,
        'F1 Weighted': f1_w,
        'F1 Macro': f1_m,
        'ROC-AUC': roc,
        'F1 Dropout': f1_dropout,
        'y_pred': y_pred,
        'y_prob': y_prob,
    }

    print(f"\n{'─'*50}")
    print(f"  {name}")
    print(f"  Accuracy:      {acc:.4f}")
    print(f"  F1 Weighted:   {f1_w:.4f}")
    print(f"  F1 Macro:      {f1_m:.4f}")
    print(f"  ROC-AUC:       {roc:.4f}")
    print(f"  F1 (Dropout):  {f1_dropout:.4f}")
    print()
    print(classification_report(y_test, y_pred,
                                 target_names=le.classes_,
                                 digits=4))




STEP 6: EVALUATION ON TEST SET

──────────────────────────────────────────────────
  Logistic Regression
  Accuracy:      0.7345
  F1 Weighted:   0.7484
  F1 Macro:      0.6958
  ROC-AUC:       0.8751
  F1 (Dropout):  0.7739

              precision    recall  f1-score   support

     Dropout     0.8487    0.7113    0.7739       284
    Enrolled     0.4059    0.6101    0.4874       159
    Graduate     0.8603    0.7941    0.8259       442

    accuracy                         0.7345       885
   macro avg     0.7050    0.7051    0.6958       885
weighted avg     0.7749    0.7345    0.7484       885


──────────────────────────────────────────────────
  Decision Tree
  Accuracy:      0.6610
  F1 Weighted:   0.6821
  F1 Macro:      0.6249
  ROC-AUC:       0.8202
  F1 (Dropout):  0.7193

              precision    recall  f1-score   support

     Dropout     0.7510    0.6901    0.7193       284
    Enrolled     0.3156    0.5220    0.3934       159
    Graduate     0.8476    0.6923    0.7

# ─────────────────────────────────────────────────────────────────────────────
# 7. VISUALISATIONS
# ─────────────────────────────────────────────────────────────────────────────

In [8]:
print("\n" + "=" * 70)
print("STEP 7: GENERATING VISUALISATIONS")
print("=" * 70)

model_names = list(results.keys())
palette = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

# ── 7a. Metrics comparison bar chart ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))
metrics = ['Accuracy', 'F1 Weighted', 'F1 Macro', 'ROC-AUC', 'F1 Dropout']
x = np.arange(len(metrics))
width = 0.18

for i, (name, color) in enumerate(zip(model_names, palette)):
    vals = [results[name][m] for m in metrics]
    bars = ax.bar(x + i * width, vals, width, label=name, color=color,
                  alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.004,
                f'{val:.3f}', ha='center', va='bottom', fontsize=7.5)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Model Performance Comparison', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('plot_metrics.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plot_metrics.png")

# ── 7b. Confusion matrices (2×2 grid) ─────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Confusion Matrices — Test Set', fontsize=15, fontweight='bold')

for ax, (name, color) in zip(axes.flatten(), zip(model_names, palette)):
    cm = confusion_matrix(y_test, results[name]['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('plot_confusion.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plot_confusion.png")

# ── 7c. ROC curves (OvR) ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('ROC Curves (One-vs-Rest) per Class', fontsize=14, fontweight='bold')

class_colors = ['#e74c3c', '#2ecc71', '#3498db']
for cls_idx, cls_name in enumerate(le.classes_):
    ax = axes[cls_idx]
    y_bin = (y_test == cls_idx).astype(int)
    for name, color in zip(model_names, palette):
        prob = results[name]['y_prob'][:, cls_idx]
        fpr, tpr, _ = roc_curve(y_bin, prob)
        auc = roc_auc_score(y_bin, prob)
        ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', color=color, linewidth=2)
    ax.plot([0,1], [0,1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_title(f'Class: {cls_name}', fontweight='bold', fontsize=12)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(fontsize=8.5)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('plot_roc.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plot_roc.png")

# ── 7d. Feature importances ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 7))
fig.suptitle('Top 15 Feature Importances', fontsize=14, fontweight='bold')

importance_models = {
    'Decision Tree':    trained_models['Decision Tree'].feature_importances_,
    'Random Forest':    trained_models['Random Forest'].feature_importances_,
    'Gradient Boosting': trained_models['Gradient Boosting'].feature_importances_,
}

feat_palette = ['#3498db', '#2ecc71', '#f39c12']
for ax, (name, imps), color in zip(axes, importance_models.items(), feat_palette):
    feat_imp = pd.Series(imps, index=X.columns).nlargest(15)
    feat_imp.sort_values().plot(kind='barh', ax=ax, color=color, alpha=0.85, edgecolor='white')
    ax.set_title(name, fontweight='bold', fontsize=12)
    ax.set_xlabel('Importance Score')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('plot_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plot_importance.png")

# ── 7e. Decision Tree visualisation ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(
    trained_models['Decision Tree'],
    feature_names=X.columns.tolist(),
    class_names=le.classes_.tolist(),
    filled=True, rounded=True, max_depth=3,
    fontsize=9, ax=ax,
    impurity=False, proportion=True
)
ax.set_title('Decision Tree Structure (depth ≤ 3 shown)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('plot_tree.png', dpi=120, bbox_inches='tight')
plt.close()
print("  Saved: plot_tree.png")

# ── 7f. CV scores boxplot ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
cv_data = [cv_scores[n] for n in model_names]
bp = ax.boxplot(cv_data, labels=model_names, patch_artist=True,
                medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], palette):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)
ax.set_title('5-Fold Cross-Validation F1 (Weighted)', fontsize=13, fontweight='bold')
ax.set_ylabel('F1 Score')
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=10)
plt.tight_layout()
plt.savefig('plot_cv.png', dpi=150, bbox_inches='tight')
plt.close()
print("  Saved: plot_cv.png")



STEP 7: GENERATING VISUALISATIONS
  Saved: plot_metrics.png
  Saved: plot_confusion.png
  Saved: plot_roc.png
  Saved: plot_importance.png
  Saved: plot_tree.png
  Saved: plot_cv.png
